# Imports

In [3]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.coordinates import FK5
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
import healpy as hp
import matplotlib.pyplot as plt
from astropy.coordinates import Angle
from astropy import units
from scipy.interpolate import RectBivariateSpline
from scipy import interpolate
import copy
import os
import sys
from tqdm.notebook import tqdm
from tqdm import tqdm
import glob
import h5py
import matplotlib
# from src2.ms_fit_joint import *
from src2.fit_funcs import *
matplotlib.rcParams['mathtext.fontset'] = 'cm'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams["font.size"] = "18"
PATH="../Sensitivty_files/Sky_spectrum/"
import scipy.io

## Load necessary files and store constants

In [4]:
sellon=-140.625
sellat=-18.209956864283015
power_output=np.load('../Sensitivty_files/power_output.npy',allow_pickle=True)

In [5]:
CHANNEL_WIDTH       = 0.06103515625/2.0 #/* MHz */
NUMBER_OF_CHANNELS  = 8193
START_FREQUENCY     = 0.00 #/* MHz */

PCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
PNUMBER_OF_CHANNELS = 41
PSTART_FREQUENCY    = 210.44921875 #/* MHz */

RCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
RNUMBER_OF_CHANNELS = 103
RSTART_FREQUENCY    = 109.86328125 #/* MHz */

QCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
QNUMBER_OF_CHANNELS = 72
QSTART_FREQUENCY    = 39.5507812500 #/* MHz */

SITE_LATITUDE       = sellat
SITE_LONGITUDE      = sellon
NHPIX               = 3072
PI                  = 3.14159265358979

TCMB                = 2.72548

filename_coord      = "PIXEL_LISTING_NESTED_R4_GALACTIC.txt" #GMOSS

filename_pix_spec1  ="total_model_spec_13jan17_3072pix_103fq.txt"
filename_pix_spec2  ="total_model_spec_24jan17_1_3072_freq_40_110_MHz_16spacing.txt"
filename_pix_spec3  ="total_model_spec_31jan17_210_250_16spacing.txt"

beam_path  = '/home/saurabhs/Documents/Yogen_starfire/Farfield1MHz/'
# rt_file    = 'gamma_linear.txt'
file_list  = sorted(glob.glob(os.path.join(beam_path,"*mhz.txt")))


##### Fitting constants

fmin = 56.0/1e3
fmax = 109.0/1e3

LOWF   = fmin*1e3
HGHF   = fmax*1e3


In [6]:
def Read_Two_Column_File(file_name):
    with open(file_name, 'r') as data:
        x = []
        y = []
        for val, line in enumerate(data):
            if val==0:
                continue
            p = line.split()
            x.append(float(p[0]))
            y.append(float(p[1]))
    return x, y

def Read_pixel_freq(file_name, nfreq):
    with open(file_name, 'r') as data:
        x = np.zeros((NHPIX, nfreq))
        for val, line in enumerate(data):
            if val==0:
                continue
            x[val-1] = [float(i) for i in line.split()]
    return x, np.shape(x)[1]

def select_freq_1d(x1, low, high): 
    prl = 0 
    prh = 0 
    for i in range(0, len(x1)):
        if x1[i]<=low:
            i_low=i+1
            prl = 1 
        if x1[i]>=high:
            i_high=i
            prh = 1 
            break
    if prl==1 and prh==0:
        i_high = len(x1)-1
    if prl==0 and prh==1:
        i_low = 1    
    return i_low, i_high


In [7]:
phi_res   = 1
theta_res = 1

phi_array   = np.arange(0, 360, phi_res)
theta_array = np.arange(-90, 90 + theta_res, theta_res) #because arange does not accept endpoint. Stupid.
freq_array  = []
file_array  = []

def get_freq_from_file(filename):
    _temp = os.path.basename(filename).replace('.txt','').replace('mhz','')
    return float(_temp)
    
for ii, file_add in enumerate(file_list):
    freq_array.append(get_freq_from_file(file_add))
    file_array.append(file_add)
    
np.testing.assert_array_equal(file_array, file_list)

freq_array = np.array(freq_array)
file_array = np.array(file_array)

isort      = np.argsort(freq_array)
freq_array = freq_array[isort]
file_array = file_array[isort]

beam_3D_unnorm = np.zeros((len(freq_array), len(theta_array), len(phi_array)))

for ii, freq in enumerate(freq_array):
    print("Processing frequency {:.1f} MHz".format(freq))
    file_add = file_array[ii]
    with open(file_add) as fa:
        for line_aa in fa.readlines()[2:]:
            line_aa = line_aa.strip()
            col1    = line_aa.split('\t')
            all_val = np.array(list(map(float, col1[0].split())))

            theta   = all_val[0] - 90
            phi     = all_val[1]
            beam    = all_val[2]
            
            iphi    = np.where(phi_array==phi)[0][0]
            itheta  = np.where(theta_array==theta)[0][0]
            beam_3D_unnorm[ii, itheta, iphi] = beam

beam_3D = np.zeros((len(freq_array), len(theta_array), len(phi_array)))
for ifreq in range(beam_3D.shape[0]):
    for iphi in range(beam_3D.shape[2]):
        beam_3D[ifreq,:,iphi] = beam_3D_unnorm[ifreq, :, iphi]/np.amax(beam_3D_unnorm[ifreq, :, iphi])

theta_array_new = theta_array
phi_array_new   = phi_array

Processing frequency 55.0 MHz
Processing frequency 56.0 MHz
Processing frequency 57.0 MHz
Processing frequency 58.0 MHz
Processing frequency 59.0 MHz
Processing frequency 60.0 MHz
Processing frequency 61.0 MHz
Processing frequency 62.0 MHz
Processing frequency 63.0 MHz
Processing frequency 64.0 MHz
Processing frequency 65.0 MHz
Processing frequency 66.0 MHz
Processing frequency 67.0 MHz
Processing frequency 68.0 MHz
Processing frequency 69.0 MHz
Processing frequency 70.0 MHz
Processing frequency 71.0 MHz
Processing frequency 72.0 MHz
Processing frequency 73.0 MHz
Processing frequency 74.0 MHz
Processing frequency 75.0 MHz
Processing frequency 76.0 MHz
Processing frequency 77.0 MHz
Processing frequency 78.0 MHz
Processing frequency 79.0 MHz
Processing frequency 80.0 MHz
Processing frequency 81.0 MHz
Processing frequency 82.0 MHz
Processing frequency 83.0 MHz
Processing frequency 84.0 MHz
Processing frequency 85.0 MHz
Processing frequency 86.0 MHz
Processing frequency 87.0 MHz
Processing

In [8]:
# gamme_file = os.path.join(beam_path, rt_file)

gamma_freq = []
gamma_val  = []

# with open(gamme_file) as fa:
#     for line_aa in fa.readlines()[3:]:
#         line_aa = line_aa.strip()
#         col1    = line_aa.split('\t')
#         _freq   = np.array(list(map(float, col1[0].split())))[0]
#         _val    = np.array(list(map(float, col1[1].split())))[0]
#         gamma_freq.append(_freq)
#         gamma_val.append(_val)

# gamma_freq = np.array(gamma_freq)
# gamma_val  = np.array(gamma_val)
        
gamma_freq = freq_array
gamma_val = 0.0*np.ones(len(gamma_freq))
    
gamma_func = scipy.interpolate.interp1d(gamma_freq, gamma_val)

In [9]:
from scipy.interpolate import RegularGridInterpolator
my_interpolating_function = RegularGridInterpolator((\
                                                     freq_array, theta_array_new, \
                                                     phi_array_new), beam_3D)

extent = (phi_array[0], phi_array[-1], theta_array[0], theta_array[-1])

# nplots = len(freq_array)
# ncol   = 3
# nrow   = int(np.ceil(nplots/ncol))

# fig, _ax = plt.subplots(nrow,ncol,figsize=(6*ncol,4*nrow))
# axs      = np.ravel(_ax)

# for val, (ax, freq) in enumerate(zip(axs, freq_array)):
#     im = ax.imshow(beam_3D[val], aspect='auto', origin='lower', \
#                    extent=extent, vmin=0, vmax=1, cmap='inferno')
#     fig.colorbar(im, ax=ax, label='Gain (abs)')
#     ax.set_title("Freq: {:.1f} MHz".format(freq))

# for ax in _ax[:,0]:
#     ax.set_ylabel("Elevation (degree)")
# for ax in _ax[-1]:
#     ax.set_xlabel("Azimuth (degree)")
    
# fig.tight_layout()
# # plt.savefig("beam_plot", dpi=100)
# plt.show()

nfreq, ntheta, nphi = beam_3D.shape
ind_phi_0 = np.argmin(np.abs(phi_array))

## GMOSS

In [10]:
ll_coordinate, bb_coordinate = np.radians(Read_Two_Column_File(PATH+filename_coord))

# T1, nspec1 = Read_pixel_freq(PATH+filename_pix_spec1, RNUMBER_OF_CHANNELS)
# T2, nspec2 = Read_pixel_freq(PATH+filename_pix_spec2, QNUMBER_OF_CHANNELS)
# T3, nspec3 = Read_pixel_freq(PATH+filename_pix_spec3, PNUMBER_OF_CHANNELS)

# freq    = np.zeros(nspec1+nspec2+nspec3)
# T_pix   = np.zeros((NHPIX, nspec1+nspec2+nspec3))
# print(np.shape(T_pix))

# for i in range(0, nspec2):
#     freq[i]=(float)(QSTART_FREQUENCY/1000.0) +\
#             ((float)(QCHANNEL_WIDTH)/1000.0)*(float)(i)
# for i in range(nspec2, (nspec1+nspec2)):
#     freq[i]=(float)(RSTART_FREQUENCY/1000.0) + ((float)(RCHANNEL_WIDTH)/1000.0)*(float)(i-nspec2)
# for i in range(nspec1+nspec2, (nspec1+nspec2+nspec3)):
#     freq[i]=(float)(PSTART_FREQUENCY/1000.0)+((float)(PCHANNEL_WIDTH)/1000.0)*(float)(i-(nspec1+nspec2))

# for j in range(0, NHPIX):
#     T_pix[j][0:nspec2]=T2[j]
#     T_pix[j][(nspec2):(nspec1+nspec2)]=T1[j]
#     T_pix[j][(nspec1+nspec2):(nspec1+nspec2+nspec3)]=T3[j]
T_pix = np.loadtxt('/home/saurabhs/Documents/gmoss_proto/gmoss_pre/GMOSS_PRATUSH.txt')
freq = np.arange(55,110,0.244)   
freq_org, T_pix_org = copy.deepcopy(freq), copy.deepcopy(T_pix)


(3072, 216)


In [11]:
ifr_low, ifr_hgh = select_freq_1d(freq_org, fmin, fmax)
freq  = freq_org[ifr_low:ifr_hgh]*1e3
T_pix = T_pix_org[:,ifr_low:ifr_hgh]

In [12]:
data21   = scipy.io.loadmat('../Sensitivty_files/Sky_spectrum/Data_18March_wMFP.mat')
data21   = data21['Data2']/1e3 
fr       = np.loadtxt('../Sensitivty_files/Sky_spectrum/freq_saras.txt') #Frequency in MHz
signal   = interpolate.interp1d(fr, data21)

# Convolve sky + RFI with beam pattern

In [13]:
NPIX = hp.nside2npix(16) # Storing the number of pixels of the map corresponding to the given NSIDE
pix=hp.ang2pix(16,sellon,sellat,lonlat=True)

NPIX = 3072 # Storing the number of pixels of the map corresponding to the given NSIDE
pix=hp.ang2pix(16,sellon,sellat,lonlat=True)

In [42]:
import h5py
filename = '/home/saurabhs/Documents/Yogen_starfire/Farfield244KHz/pratush_space.h5'
files=h5py.File(filename)
temps=files['T_A'][:]

In [43]:
np.sum(np.isnan(temps))

113000

## Add system noise

In [16]:
# tau=158.69*3600
# sigma=np.zeros((5,55))
# for i in range(5):
#     for j in range(55):
#         sigma[i,j]=(70+T_monopole_AUS_sin[i,j])/(np.sqrt((1e6)*tau))
#         #T_monopole_AUS_sin[i,j]+=np.random.normal(0,1,1)*sigma[i,j]

## Residuals for null hypothesis

In [30]:
freq_null=copy.deepcopy(freq)

NameError: name 'copy' is not defined

In [31]:
avg_spectra = np.mean(temps, axis=0) 
# Add radiometer noise

# flags=avg_spectra>1e4
# avg_spectra=avg_spectra[~flags]
# freq_null=freq_null[~flags]
# freq_null = np.asarray(freq_null, dtype=np.float64)
avg_spectra = np.asarray(avg_spectra, dtype=np.float64)
e=np.ones_like(freq_null)

# Codes taken from investigation_cleaner.ipynb

for i in range(3,11):
    print(i)
    initial_ = np.zeros(i, dtype='float128')
    if i ==3:
        initial_ = np.polyfit(y= np.log10(avg_spectra), x = rescale(freq_null), deg = 2)
    if i > 3:
        if para_new[0] ==0:
            p_order = len(para_new)-2
            break
        initial_[1:] = np.array(para_new, dtype='float128')
    para_new,res_cons,fit_curve_mag, chi_mag = chi_dunk(x1=freq_null,y1=avg_spectra,\
                                             p00=initial_, gloss = True, keju_inf=0,\
                                               e1=e,iterations=50)
    

NameError: name 'freq_null' is not defined

In [19]:
rms_null = np.sqrt(np.mean(res_cons**2))
print(rms_null)

0.0003772911836768928


## Inject 21 cm signal

In [20]:
data21   = scipy.io.loadmat('../Sensitivty_files/Sky_spectrum/Data_18March_wMFP.mat')
data21   = data21['Data2']/1e3 
fr       = np.loadtxt('../Sensitivty_files/Sky_spectrum/freq_saras.txt') #Frequency in MHz
signal   = interpolate.interp1d(fr, data21)
T21=  signal(freq)[100] #max 263 
freq_alt=copy.deepcopy(freq)

In [21]:
avg_spectra1 = np.mean(T_monopole_AUS_sin, axis=0)+T21

# flags=avg_spectra1>1e4

# avg_spectra1=avg_spectra1[~flags]
# freq_alt=freq_alt[~flags]
# freq_alt = np.asarray(freq_alt, dtype=np.float64)
avg_spectra1 = np.asarray(avg_spectra1, dtype=np.float64)
e=np.ones_like(freq_alt)

# Codes taken from investigation_cleaner.ipynb

for i in range(3,11):
    print(i)
    initial_ = np.zeros(i, dtype='float128')
    if i ==3:
        initial_ = np.polyfit(y= np.log10(avg_spectra1), x = rescale(freq_alt), deg = 2)
    if i > 3:
        if para_new1[0] ==0:
            p_order = len(para_new1)-2
            break
        initial_[1:] = np.array(para_new1, dtype='float128')
    para_new1,res_cons1,fit_curve_mag1, chi_mag1 = chi_dunk(x1=freq_alt,y1=avg_spectra1,\
                                             p00=initial_, gloss = True, keju_inf=0,\
                                               e1=e,iterations=50)
    #fit_curves_array = np.vstack((fit_curves_array,fit_curve_))
    #print(np.std(res_ns))

3
initial_6


  0%|          | 0/50 [00:00<?, ?it/s]

Coefficients : [-1.54080083e-03 -3.55490137e-01  3.27916218e+00] 

Residual rms : 0.10961382268937259
4
initial_6


  0%|          | 0/50 [00:00<?, ?it/s]

Coefficients : [ 1.54002222e-04 -1.48433678e-03 -3.55575610e-01  3.27914453e+00] 

Residual rms : 0.02427275155707873
5
initial_6


...

Coefficients : [-4.20112369e-07  1.08519328e-05 -5.52089069e-05  1.20718547e-04
 -1.43857687e-03 -3.55560571e-01  3.27914058e+00] 

Residual rms : 0.009999736260442139
8
initial_6


  0%|          | 0/50 [00:00<?, ?it/s]

Coefficients : [ 6.00163981e-08 -4.20114859e-07  1.06217396e-05 -5.52095651e-05
  1.20922587e-04 -1.43857508e-03 -3.55560614e-01  3.27914058e+00] 

Residual rms : 0.009980897993130249
9
initial_6


  0%|          | 0/50 [00:00<?, ?it/s]

Coefficients : [ 0.00000000e+00  6.00163982e-08 -4.20114859e-07  1.06217396e-05
 -5.52095651e-05  1.20922587e-04 -1.43857508e-03 -3.55560614e-01
  3.27914058e+00] 

Residual rms : 0.009980897992965277
10


In [26]:
excess_rms = np.sqrt(np.std(avg_spectra1-fit_curve_mag1)**2-np.std(avg_spectra-fit_curve_mag)**2)
print(excess_rms, excess_rms/np.std(avg_spectra-fit_curve_mag))

0.009973761673911355 26.43518350249999


In [1]:
plt.plot(freq_null, (avg_spectra-fit_curve_mag)*1000, alpha=0.6,label="No signal")
plt.plot(freq_null, (avg_spectra1-fit_curve_mag1)*1000, alpha=0.6,label="With signal")
plt.plot(freq,T21*1000)
plt.legend()
plt.ylabel('residuals (mK)')
plt.xlabel('Frequency (MHz)')
plt.show()

NameError: name 'plt' is not defined

## Run for each model

In [25]:
check_smoothness(para_new,freq_null,keju_inf=1)

True